# Real vs Fake Faces

We are going to use `Fake-Vs-Real-Faces` dataset. The Dataset contains two types of image which are: `Fake faces` image collected from the website [thispersondoesnotexist.com](thispersondoesnotexist.com). , `Real faces` images are re collected through the API of the website [Unsplash](https://unsplash.com/) and then the faces are cropped out of using OpenCV library.

Here some additional information about the dataset:
- Total number of images: 1288
- Number of "Fake" faces: 700
- Number of "Real" faces: 589

### 1. Import Libraries 

In [1]:
import torch 
import torch.nn as nn
from torch.nn import Linear, Dropout, ReLU
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, random_split, Dataset
from torchvision.datasets import ImageFolder # Stream data from images stored in folders
import torch.optim as optim
import torch.nn.functional as F
# from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score
from torchinfo import summary

from sklearn.model_selection import StratifiedKFold, train_test_split
import os # allow s to access files
import numpy as np
from PIL import Image # Allows us to Load Images
from collections import Counter # Utility function to give us the counts of unique items in an iterable
import matplotlib.pyplot as plt
import shutil

## 2. Reconstruct the Folder structure of the Data

In [2]:
torch.manual_seed(42)

Before loading, let's reconstruct the data folder structure to give us comfort when using the `ImageFolder` class

In [3]:
def copy_image(des_dir, split_name, cls, file_list, cls_path):

    """
    des_dir: data_split
    split_name: can be "train" or "test"
    cls and cls_path: is class which can be "fake" or "real"
    file_list: list of images files

    save a copy of images from folder 'cls_path/img' into folder called 'des_dir/split_name/cls/img' 
    """
    
    for img in file_list:
        src = os.path.join(cls_path, img)
        dest = os.path.join(des_dir, split_name, cls, img)
        shutil.copy2(src, dest)

In [4]:

def split_and_save_to_disk(src_dir, des_dir):

    """
    reconstruct the folder structure 
    from:
        data/
            fake/
            real

    to:
        data_split/
            train/
                fake/
                real/
            test/
                fake/
                real/
    
    """
    
    # # splite ratios
    train_ratio = 0.8
    test_ratio = 0.2

    if os.path.exists(des_dir):
        print(f"{src_dir} already exists")
    else:
        print("reconstruct the data folder structure")

        # create folder 
        for split in ['train', 'test']:
            for cls in os.listdir(src_dir):
                os.makedirs(os.path.join(des_dir, split, cls), exist_ok=True)  # make it similar to this (des_dir/train/fake) and  (des_dir/train/real)


        # Process each class folder 
        for cls in os.listdir(src_dir):
            cls_path = os.path.join(src_dir, cls)
            if not os.path.isdir(cls_path):
                continue


            images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            # split train and (test and val)
            train_data, test_data = train_test_split(images, test_size=test_ratio, random_state=42)


            # splitting the data 
            print(f"splitting class '{cls}':\n train:{len(train_data)}\n test:{len(test_data)}")

            # des_dir, split_name, cls, file_list, cls_path
            copy_image(des_dir, "train", cls, train_data, cls_path)
            copy_image(des_dir, "test", cls, test_data, cls_path)
            
        print("Copying proces is finished")
        

## 3. Load and Prepocess the data

First Let's define the transforms pipeline for `data augmentatin` and `preprocessing` steps

In [5]:
split_and_save_to_disk("data", "data_split")

data already exists


In [6]:
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# for data preprocessing and data augmentation for trianing data
train_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


test_valid_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),

])

Let's load our images using `ImageFolder`. This class inherits from `DatasetFolder`  class. So, we can override the classes to create custom datasets as well. Because we need to custome dataset class becasue `ImageFolder` apply a single tansform on the whole dataset while we need to apply custome `transform` on each subset in the dataset.

In [7]:
path_to_train = './split_data/train'
path_to_test = './split_data/test'

train_data = ImageFolder(root=path_to_train, transform=None)
test_data = ImageFolder(root=path_to_test, transform=test_valid_transforms)

class CustomSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        
        actual_idx = self.subset.indices[idx]
        
        image_path, label = self.subset.dataset.samples[actual_idx]
        image = self.subset.dataset.loader(image_path) 
        
        
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        
        return image, label
    
    def __len__(self):
        return len(self.subset)
        
# Create the DataLoders
BATCH_SIZE = 16
# train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

## 3. Build Helper functions

Before starting building the model we will build a helper function to save our models

In [8]:
def save_model(epochs, model, optimizer, loss_fun, model_name):

    """
    Function to save the trained model to disk
    """
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_fun,
    }, f'outputs/{model_name}.pth')

The next function is for saving the loss and accuracy plots after the training completes

In [9]:
def save_plots(train_acc, valid_acc, test_acc, train_loss, valid_loss, test_loss):

    """
    Function to save the loss and accurcy plots to disk
    """

    # accuracy plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_acc, color='green', linestyle='-', label='train accuracy')
    plt.plot(valid_acc, color='red', linestyle='-', label='validation accuracy')
    plt.plot(test_acc, color='cyan', linestyle='-', label='test accuracy')
    plt.xlabel('EPOCHs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('outputs/accuracy.png')

    # loss plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, color='orange', linestyle='-', label='train loss')
    plt.plot(valid_loss, color='blue', linestyle='-', label='valid loss')
    plt.plot(test_loss, color='magenta', linestyle='-', label='test loss')
    plt.xlabel('EPOCHs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('outputs/loss.png')

## 4. Prepparing the Data Loaders and Stratified KFold

Now we will build our `data loader` and `stratified KFold` spliter for that we need for training.

In [10]:
N_SPLITs = 5
SEED = 42

In [11]:
def train(model, training_data, X, y, optimizer, loss_fn, Epochs = 10, N_SPLITs = 4):

    """
    Train the given model
    """


    
    skf = StratifiedKFold(n_splits=N_SPLITs, shuffle=True, random_state=SEED)
    stratified_scores = []

    # folds results 
    all_fold_results = []

    # Lists to collect metrics
    train_acc_list, valid_acc_list = [], []
    train_loss_list, valid_loss_list = [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):



        # Initialize fold-specific metrics
        fold_train_loss = []
        fold_val_loss = []
        fold_train_acc = []
        fold_val_acc = []
        
        # Create the Data Loaders
        train_subset = Subset(training_data, train_idx)
        val_subset = Subset(training_data, valid_idx)


        # Apply transforms
        train_data_fold = CustomSubset(train_subset, transform=train_transforms)
        val_data_fold = CustomSubset(val_subset, transform=test_valid_transforms)
    
        # Dataloaders
        train_loader = DataLoader(train_data_fold, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(val_data_fold, batch_size=BATCH_SIZE, shuffle=False)

        ### Training ###
        model.train() 
        print("Training")
        for epoch in range(Epochs):

            train_loss, correct, total = 0.0, 0, 0
            
            for images, labels in train_loader:
                predictions = model(images)
                labels = labels.unsqueeze(1).float()
                loss = loss_fn(predictions, labels)
                loss.backward() # backpagation

                train_loss += loss.item()
                
                optimizer.step() # update the weight
                optimizer.zero_grad() # set the gradient to zero
                
                preds = (torch.sigmoid(predictions) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)


            # calculate the accuracy and loss for the current epoch
            train_accuracy = correct / total
            epoch_train_loss = train_loss / len(train_loader)


            ### Validation ###
            model.eval()
            print("Evaluting")
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in valid_loader:
                    predictions = model(images)
                    labels = labels.unsqueeze(1).float()
                    val_loss += loss_fn(predictions, labels).item()
                    
                    preds = (torch.sigmoid(predictions) > 0.5).float()
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

                    
            val_accuracy = correct / total
            epoch_val_loss = val_loss / len(valid_loader)
     

        
            fold_train_loss.append(train_loss)
            fold_val_loss.append(val_loss)
            fold_train_acc.append(train_accuracy)
            fold_val_acc.append(val_accuracy)
            
            print(f"Epoch {epoch+1}/{Epochs} - Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

            
        # Store metrics for this fold
        all_fold_results.append({
            'fold': fold + 1,
            'train_loss': fold_train_loss,
            'val_loss': fold_val_loss,
            'train_acc': fold_train_acc,
            'val_acc': fold_val_acc
        })


    avg_val_acc = sum(r['val_acc'][-1] for r in all_fold_results) / N_SPLITs
    avg_val_loss = sum(r['val_loss'][-1] for r in all_fold_results) / N_SPLITs
    print(f"\nAverage Validation Accuracy across folds: {avg_val_acc:.4f}")
    print(f"Average Validation Loss across folds: {avg_val_loss:.4f}")

    return all_fold_results
        
              
    

## 4, 5. Build and Train the Models

> ### 4.1 InceptionV3

In [13]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import resnet50
from torchvision.models import ResNet50_Weights

# initialize the pretrained ResNet50
restnet_transfer = resnet50(weights=ResNet50_Weights.DEFAULT)

restnet_transfer.eval()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\AHMEDELNIEL/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:29<00:00, 3.53MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

> ### 4.2 Freeze early layers 

In [24]:
# for parameter in model.parameters():
#     parameter.requires_grad = False

for parameter in restnet_transfer.parameters():
    parameter.requires_grad = False


> ### 4.3 Modfiy the inceptionv3 (output layer)

In [25]:
restnet_transfer.fc = nn.Sequential(
    Linear(in_features=2048, out_features=1, bias=True),
)

> ### 4.4 Get model summary

In [29]:
summary(restnet_transfer,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
ResNet (ResNet)                          [16, 3, 224, 224]  [16, 1]            --                 Partial
├─Conv2d (conv1)                         [16, 3, 224, 224]  [16, 64, 112, 112] (9,408)            False
├─BatchNorm2d (bn1)                      [16, 64, 112, 112] [16, 64, 112, 112] (128)              False
├─ReLU (relu)                            [16, 64, 112, 112] [16, 64, 112, 112] --                 --
├─MaxPool2d (maxpool)                    [16, 64, 112, 112] [16, 64, 56, 56]   --                 --
├─Sequential (layer1)                    [16, 64, 56, 56]   [16, 256, 56, 56]  --                 False
│    └─Bottleneck (0)                    [16, 64, 56, 56]   [16, 256, 56, 56]  --                 False
│    │    └─Conv2d (conv1)               [16, 64, 56, 56]   [16, 64, 56, 56]   (4,096)            False
│    │    └─BatchNorm2d (bn1)            [16, 64, 56, 56]   [16,

> ### 4.4 Train the Model

In [35]:
# Specifiy learing rate and number of Epochs
restnet_transfer_LR = 0.001
restnet_transfer_Epochs = 2
restnet_transfer_n_splits = 4
# Define loss and optimizer
restnet_transfer_loss_fun = nn.BCEWithLogitsLoss()
restnet_transfer_optimizer = torch.optim.Adam(restnet_transfer.parameters(), lr=restnet_transfer_LR)


train_X = [train_data.samples[img][0] for img in range(len(train_data.samples))]
train_y = [train_data.samples[label][1] for label in range(len(train_data.samples))]

skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
BATCH_SIZE = 16

In [36]:
resent50_transfer_train_result = train(restnet_transfer, train_data, train_X, train_y, restnet_transfer_optimizer, restnet_transfer_loss_fun, restnet_transfer_Epochs, N_SPLITs = restnet_transfer_n_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.4890, Train Acc: 0.9094, Val Loss: 0.3744, Val Acc: 0.8760
Evaluting
Epoch 2/2 - Train Loss: 0.2975, Train Acc: 0.9547, Val Loss: 0.2180, Val Acc: 0.9806
Training
Evaluting
Epoch 1/2 - Train Loss: 0.2373, Train Acc: 0.9715, Val Loss: 0.1974, Val Acc: 0.9729
Evaluting
Epoch 2/2 - Train Loss: 0.1676, Train Acc: 0.9832, Val Loss: 0.1685, Val Acc: 0.9729
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1787, Train Acc: 0.9638, Val Loss: 0.1284, Val Acc: 0.9729
Evaluting
Epoch 2/2 - Train Loss: 0.1370, Train Acc: 0.9690, Val Loss: 0.1077, Val Acc: 0.9884
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1371, Train Acc: 0.9755, Val Loss: 0.1340, Val Acc: 0.9572
Evaluting
Epoch 2/2 - Train Loss: 0.1024, Train Acc: 0.9819, Val Loss: 0.1116, Val Acc: 0.9767

Average Validation Accuracy across folds: 0.9796
Average Validation Loss across folds: 2.5747


> ### 5.1 InceptionV3 partial

In [37]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import resnet50
from torchvision.models import ResNet50_Weights

# initialize the pretrained ResNet50
restnet_full = resnet50(weights=ResNet50_Weights.DEFAULT)

restnet_full.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

> ### 5.2 Freeze early layers 

In [38]:
for parameter in restnet_full.parameters():
    parameter.requires_grad = True

> ### 5.3 Get model summary

In [39]:
summary(restnet_full,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
ResNet (ResNet)                          [16, 3, 224, 224]  [16, 1000]         --                 True
├─Conv2d (conv1)                         [16, 3, 224, 224]  [16, 64, 112, 112] 9,408              True
├─BatchNorm2d (bn1)                      [16, 64, 112, 112] [16, 64, 112, 112] 128                True
├─ReLU (relu)                            [16, 64, 112, 112] [16, 64, 112, 112] --                 --
├─MaxPool2d (maxpool)                    [16, 64, 112, 112] [16, 64, 56, 56]   --                 --
├─Sequential (layer1)                    [16, 64, 56, 56]   [16, 256, 56, 56]  --                 True
│    └─Bottleneck (0)                    [16, 64, 56, 56]   [16, 256, 56, 56]  --                 True
│    │    └─Conv2d (conv1)               [16, 64, 56, 56]   [16, 64, 56, 56]   4,096              True
│    │    └─BatchNorm2d (bn1)            [16, 64, 56, 56]   [16, 64, 56,

> ### 5.4 Modfiy the InceptionV3 (output layer)

In [40]:
restnet_full.fc = nn.Sequential(
    Linear(in_features=2048, out_features=1, bias=True),
)

> ### 5.5 Train the model

In [41]:
# Specifiy learing rate and number of Epochs
restnet_full_LR = 0.001
restnet_full_Epochs = 2
restnet_full_splits = 4
# Define loss and optimizer
restnet_full_loss_fun = nn.BCEWithLogitsLoss()
restnet_full_optimizer = torch.optim.Adam(restnet_full.parameters(), lr=restnet_full_LR)

In [42]:
result_train_full_model = train(restnet_full, train_data, train_X, train_y, restnet_full_optimizer, restnet_full_loss_fun, restnet_full_Epochs, restnet_full_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.1645, Train Acc: 0.9340, Val Loss: 1.2606, Val Acc: 0.8411
Evaluting
Epoch 2/2 - Train Loss: 12.7385, Train Acc: 0.5278, Val Loss: 0.7012, Val Acc: 0.5426
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1251, Train Acc: 0.9534, Val Loss: 0.0083, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.6458, Train Acc: 0.6080, Val Loss: 0.6909, Val Acc: 0.5426
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1043, Train Acc: 0.9728, Val Loss: 0.0110, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.5099, Train Acc: 0.7529, Val Loss: 0.6938, Val Acc: 0.5000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0685, Train Acc: 0.9832, Val Loss: 0.0136, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.5393, Train Acc: 0.7132, Val Loss: 0.7028, Val Acc: 0.5447

Average Validation Accuracy across folds: 0.5325
Average Validation Loss across folds: 11.8518


### 6.1 InceptionV3 partial fine-tuning

In [43]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import resnet50
from torchvision.models import ResNet50_Weights

# initialize the pretrained ResNet50
restnet_partial = resnet50(weights=ResNet50_Weights.DEFAULT)

restnet_partial.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

### 6.2 Frezze Early layers

In [45]:
for parameter in restnet_partial.parameters():
    parameter.requires_grad = False


weight_params = [
    (name, param)
    for name, param in restnet_partial.named_parameters()
    if name.endswith("weight")
]

for name, param in weight_params[-8:]:
    param.requires_grad = True
    print(f"Unfrozen weight: {name}")

Unfrozen weight: layer4.1.bn3.weight
Unfrozen weight: layer4.2.conv1.weight
Unfrozen weight: layer4.2.bn1.weight
Unfrozen weight: layer4.2.conv2.weight
Unfrozen weight: layer4.2.bn2.weight
Unfrozen weight: layer4.2.conv3.weight
Unfrozen weight: layer4.2.bn3.weight
Unfrozen weight: fc.weight


### 6.3 Modify the head of inceptionV3 

In [46]:
restnet_partial.fc = nn.Sequential(
    Linear(in_features=2048, out_features=1, bias=True),
)

### 6.4 Get the summary of the model

In [47]:
summary(restnet_partial,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
ResNet (ResNet)                          [16, 3, 224, 224]  [16, 1]            --                 Partial
├─Conv2d (conv1)                         [16, 3, 224, 224]  [16, 64, 112, 112] (9,408)            False
├─BatchNorm2d (bn1)                      [16, 64, 112, 112] [16, 64, 112, 112] (128)              False
├─ReLU (relu)                            [16, 64, 112, 112] [16, 64, 112, 112] --                 --
├─MaxPool2d (maxpool)                    [16, 64, 112, 112] [16, 64, 56, 56]   --                 --
├─Sequential (layer1)                    [16, 64, 56, 56]   [16, 256, 56, 56]  --                 False
│    └─Bottleneck (0)                    [16, 64, 56, 56]   [16, 256, 56, 56]  --                 False
│    │    └─Conv2d (conv1)               [16, 64, 56, 56]   [16, 64, 56, 56]   (4,096)            False
│    │    └─BatchNorm2d (bn1)            [16, 64, 56, 56]   [16,

In [48]:
# Specifiy learing rate and number of Epochs
restnet_partial_LR = 0.001
restnet_partial_Epochs = 2
inceptionv3_partial_splits = 4
# Define loss and optimizer
restnet_partial_loss_fun = nn.BCEWithLogitsLoss()
restnet_partial_optimizer = torch.optim.Adam(restnet_partial.parameters(), lr=restnet_partial_LR)

In [50]:
restnet_partial_partial_model = train(restnet_partial, train_data, train_X, train_y, restnet_partial_optimizer, restnet_partial_loss_fun, restnet_partial_Epochs, inceptionv3_partial_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.0626, Train Acc: 0.9871, Val Loss: 0.0317, Val Acc: 0.9922
Evaluting
Epoch 2/2 - Train Loss: 0.0538, Train Acc: 0.9845, Val Loss: 0.0116, Val Acc: 0.9961
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0500, Train Acc: 0.9884, Val Loss: 0.0077, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.0248, Train Acc: 0.9935, Val Loss: 0.0026, Val Acc: 1.0000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0358, Train Acc: 0.9871, Val Loss: 0.0017, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.0169, Train Acc: 0.9935, Val Loss: 0.0078, Val Acc: 1.0000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1236, Train Acc: 0.9871, Val Loss: 0.0120, Val Acc: 0.9961
Evaluting
Epoch 2/2 - Train Loss: 0.0243, Train Acc: 0.9935, Val Loss: 0.0143, Val Acc: 0.9961

Average Validation Accuracy across folds: 0.9981
Average Validation Loss across folds: 0.1540


In [52]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score

def test(model, test_loader, loss_fn):
    """
    Test the model and return testing metrics
    """
    model.eval()
    print("Testing")
    
    val_loss, correct, total = 0.0, 0, 0
    y_pred, y_true = [], []
    
    # Initialize metrics
    precision = BinaryPrecision()
    recall = BinaryRecall()
    f1 = BinaryF1Score()
    
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = model(images)
            labels = labels.unsqueeze(1).float()
            
   
            val_loss += loss_fn(predictions, labels).item()
            

            preds = (torch.sigmoid(predictions) > 0.5).float()
            
            y_pred.append(preds)
            y_true.append(labels)
            
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    y_pred = torch.cat(y_pred, dim=0)
    y_true = torch.cat(y_true, dim=0)
    
    # Metrics
    test_accuracy = correct / total
    test_loss = val_loss / len(test_loader)
    test_precision = precision(y_pred, y_true).item()
    test_recall = recall(y_pred, y_true).item()
    test_f1 = f1(y_pred, y_true).item()
    
    return {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1': test_f1,
    }

In [53]:
restnet_transfer_test_results = test(restnet_transfer, test_loader, restnet_transfer_loss_fun)

Testing


In [54]:
restnet_full_test_results = test(restnet_full, test_loader, restnet_full_loss_fun)

Testing


In [55]:
restnet_partial_test_results = test(restnet_partial, test_loader, restnet_partial_loss_fun)

Testing


In [56]:
restnet_transfer_test_results

{'test_loss': 0.11023484253221089,
 'test_accuracy': 0.9767441860465116,
 'test_precision': 1.0,
 'test_recall': 0.9491525292396545,
 'test_f1': 0.9739130139350891}

In [57]:
restnet_full_test_results

{'test_loss': 0.7057326038678488,
 'test_accuracy': 0.5426356589147286,
 'test_precision': 0.0,
 'test_recall': 0.0,
 'test_f1': 0.0}

In [58]:
restnet_partial_test_results

{'test_loss': 0.0010304385478876713,
 'test_accuracy': 1.0,
 'test_precision': 1.0,
 'test_recall': 1.0,
 'test_f1': 1.0}

In [61]:
avg_val_acc = sum(r['val_acc'][-1] for r in resent50_transfer_train_result) / restnet_full_splits
avg_val_loss = sum(r['val_loss'][-1] for r in resent50_transfer_train_result) / restnet_full_splits
avg_train_acc = sum(r['train_acc'][-1] for r in resent50_transfer_train_result) / restnet_full_splits
avg_train_loss = sum(r['train_loss'][-1] for r in resent50_transfer_train_result) / restnet_full_splits
print(f"Resnet50 transfer average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"Resnet50 transfer average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"Resnet50 transfer average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"Resnet50 transfer average Traing Loss across folds: {avg_train_loss:.4f}")

Resnet50 transfer average Validation Accuracy across folds: 0.9796
Resnet50 transfer average Validation Loss across folds: 2.5747
Resnet50 transfer average Training Accuracy across folds: 0.9722
Resnet50 transfer average Traing Loss across folds: 8.6303


In [62]:
avg_val_acc = sum(r['val_acc'][-1] for r in result_train_full_model) / restnet_full_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result_train_full_model) / restnet_full_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result_train_full_model) / restnet_full_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result_train_full_model) / restnet_full_splits
print(f"Resnet50 full fine-tuning average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"Resnet50 full fine-tuning  average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"Resnet50 full fine-tuning  average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"Resnet50 full fine-tuning  average Traing Loss across folds: {avg_train_loss:.4f}")

Resnet50 full fine-tuning average Validation Accuracy across folds: 0.5325
Resnet50 full fine-tuning  average Validation Loss across folds: 11.8518
Resnet50 full fine-tuning  average Training Accuracy across folds: 0.6505
Resnet50 full fine-tuning  average Traing Loss across folds: 176.8116


In [63]:
avg_val_acc = sum(r['val_acc'][-1] for r in restnet_partial_partial_model) / restnet_full_splits
avg_val_loss = sum(r['val_loss'][-1] for r in restnet_partial_partial_model) / restnet_full_splits
avg_train_acc = sum(r['train_acc'][-1] for r in restnet_partial_partial_model) / restnet_full_splits
avg_train_loss = sum(r['train_loss'][-1] for r in restnet_partial_partial_model) / restnet_full_splits
print(f"Resnet50 partial fine-tuning average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"Resnet50 partial fine-tuning  average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"Resnet50 partial fine-tuning  average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"Resnet50 partial fine-tuning  average Traing Loss across folds: {avg_train_loss:.4f}")

Resnet50 partial fine-tuning average Validation Accuracy across folds: 0.9981
Resnet50 partial fine-tuning  average Validation Loss across folds: 0.1540
Resnet50 partial fine-tuning  average Training Accuracy across folds: 0.9913
Resnet50 partial fine-tuning  average Traing Loss across folds: 1.4677
